<a href="https://colab.research.google.com/github/AIAtlantis/attention-architecture-explore/blob/main/Copy_of_MCP_GenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install Node.js (needed for Airbnb MCP server)
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - 2>/dev/null
!sudo apt-get install -y nodejs 2>/dev/null | tail -1

# Verify
!echo "Node: $(node --version) | npm: $(npm --version) | npx: $(npx --version)"

2026-02-28 19:26:49 - Installing pre-requisites
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,302 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,643 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 ht

In [ ]:
# Cell 2: Install all Python dependencies
!pip install langchain-google-genai mcp-use python-dotenv nest_asyncio mcp-server-fetch -q

print("✅ All packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.8/186.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.7/198.7 kB 14.0 MB/s eta 0:00:00
✅ All packages installed!


In [ ]:
# Cell 3: Load GROQ and Gemini API keys from Colab Secrets
import os
from google.colab import userdata

os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')

print(f"✅ Gemini API Key loaded: {os.environ['GOOGLE_API_KEY'][:8]}...")

✅ Gemini API Key loaded: AIzaSyCu...


In [ ]:
# Cell 4: Create MCP server configuration
import json

mcp_config = {
    "mcpServers": {
        "fetch": {
            "command": "python",
            "args": ["-m", "mcp_server_fetch"]
        },
        "airbnb": {
            "command": "npx",
            "args": ["-y", "@openbnb/mcp-server-airbnb"]
        }
    }
}

with open('browser_mcp.json', 'w') as f:
    json.dump(mcp_config, f, indent=2)

print("✅ browser_mcp.json created with Fetch + Airbnb servers!")

✅ browser_mcp.json created with Fetch + Airbnb servers!


In [ ]:
# Cell 5: Import libraries and apply Colab fixes
import asyncio
import os
import io
import sys
import subprocess
import warnings
import nest_asyncio

# Fix Colab's event loop
nest_asyncio.apply()

# Fix Colab's stdin/stdout for MCP subprocess spawning
sys.stdin = open(os.devnull, 'r')
sys.__stdin__ = sys.stdin

_OrigPopen = subprocess.Popen

class _ColabSafePopen(_OrigPopen):
    def __init__(self, args, **kwargs):
        for stream_name in ('stdin', 'stdout', 'stderr'):
            val = kwargs.get(stream_name)
            if val is not None \
               and val not in (subprocess.PIPE, subprocess.DEVNULL, subprocess.STDOUT) \
               and not isinstance(val, int):
                try:
                    val.fileno()
                except (io.UnsupportedOperation, OSError, AttributeError):
                    kwargs[stream_name] = subprocess.PIPE
        super().__init__(args, **kwargs)

subprocess.Popen = _ColabSafePopen

# Silence noisy warnings
warnings.filterwarnings("ignore", message=r"unclosed transport.*", category=ResourceWarning)

from langchain_google_genai import ChatGoogleGenerativeAI
from mcp_use import MCPAgent, MCPClient

print("✅ All libraries imported and Colab patches applied!")

2026-02-28 19:29:07,965 - mcp_use.telemetry.telemetry - INFO - Anonymized telemetry enabled. Set MCP_USE_ANONYMIZED_TELEMETRY=false to disable.


INFO:mcp_use.telemetry.telemetry:Anonymized telemetry enabled. Set MCP_USE_ANONYMIZED_TELEMETRY=false to disable.


✅ All libraries imported and Colab patches applied!


In [ ]:
# Cell 6: Define ThrottledMCPClient (handles rate limits)
class ThrottledMCPClient(MCPClient):
    """Custom MCP client with exponential backoff for rate limits."""

    async def call_tool(self, name: str, *args, **kwargs):
        delay = 1.0
        for attempt in range(5):
            try:
                result = await super().call_tool(name, *args, **kwargs)
                await asyncio.sleep(1.5)
                return result
            except Exception as exc:
                msg = str(exc).lower()
                if "anomaly" in msg or "429" in msg:
                    print(f"  ⚠️ [Rate limit - backoff {delay:.1f}s]")
                    await asyncio.sleep(delay)
                    delay *= 2
                    continue
                raise

print("✅ ThrottledMCPClient defined!")

✅ ThrottledMCPClient defined!


In [ ]:
# Cell 7: Define the agent runner function (Gemini 2.5 Flash Lite)
async def run_agent_query(user_query: str) -> str:
    """Run a single query through the MCP agent."""

    config_file = "browser_mcp.json"

    print("🔄 Initializing MCP client and LLM...")
    client = ThrottledMCPClient.from_config_file(config_file)
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

    agent = MCPAgent(
        llm=llm,
        client=client,
        max_steps=15,
        memory_enabled=True,
    )

    print(f"🚀 Running agent with query: {user_query}")
    print("-" * 50)

    try:
        response = await agent.run(user_query)
        return response
    except Exception as exc:
        print(f"\n🔴 ERROR DETAILS:")
        print(f"   Type:    {type(exc).__name__}")
        print(f"   Message: {exc}")
        return f"❌ Error: {exc}"
    finally:
        if hasattr(client, "close_all_sessions"):
            try:
                await client.close_all_sessions()
            except Exception:
                pass

print("✅ run_agent_query() defined with Gemini 2.5 Flash Lite!")

✅ run_agent_query() defined with Gemini 2.5 Flash Lite!


In [ ]:
# Cell 8: Test the agent - Fetch Wikipedia page
query = "Fetch the webpage https://en.wikipedia.org/wiki/Tokyo and summarize Tokyo's population and top 5 tourist attractions"
print(f"📝 Query: {query}\n")

result = await run_agent_query(query)

print(f"\n{'='*50}")
print(f"🤖 Agent Response:\n")
print(result)

📝 Query: Fetch the webpage https://en.wikipedia.org/wiki/Tokyo and summarize Tokyo's population and top 5 tourist attractions

🔄 Initializing MCP client and LLM...
🚀 Running agent with query: Fetch the webpage https://en.wikipedia.org/wiki/Tokyo and summarize Tokyo's population and top 5 tourist attractions
--------------------------------------------------
2026-02-28 19:29:52,041 - mcp_use - INFO - 🚀 Initializing MCP agent and connecting to services...


INFO:mcp_use:🚀 Initializing MCP agent and connecting to services...


2026-02-28 19:29:52,044 - mcp_use - INFO - 🔌 Found 0 existing sessions


INFO:mcp_use:🔌 Found 0 existing sessions


2026-02-28 19:29:52,047 - mcp_use - INFO - 🔄 No active sessions found, creating new ones...


...

2026-02-28 19:30:27,236 - mcp_use - INFO - ✅ Agent finished with output


INFO:mcp_use:✅ Agent finished with output


2026-02-28 19:30:27,240 - mcp_use - INFO - 🎉 Agent execution complete in 35.20 seconds


INFO:mcp_use:🎉 Agent execution complete in 35.20 seconds



🤖 Agent Response:

The population of Tokyo is 14,254,039 as of May 2025.

Here are Tokyo's top 5 tourist attractions:

1.  **Shibuya Crossing:** Known as the world's busiest intersection, it's a mesmerizing spectacle of organized chaos with thousands of pedestrians crossing simultaneously.
2.  **Senso-ji Temple:** Tokyo's oldest temple, located in Asakusa, offers a glimpse into traditional Japanese culture with its vibrant Nakamise-dori market leading up to the main hall.
3.  **Tokyo Skytree:** This towering structure provides breathtaking panoramic views of the city and beyond.
4.  **Meiji Jingu Shrine:** A peaceful oasis dedicated to Emperor Meiji and Empress Shoken, surrounded by a lush forest.
5.  **Shinjuku Gyoen National Garden:** A beautiful and diverse garden featuring French, English, and traditional Japanese landscape styles.


In [ ]:
# Cell 9: Test the agent - Fetch a tech news page
query2 = "Fetch https://news.ycombinator.com and tell me the top 5 stories on Hacker News right now"
print(f"📝 Query: {query2}\n")

result2 = await run_agent_query(query2)

print(f"\n{'='*50}")
print(f"🤖 Agent Response:\n")
print(result2)

📝 Query: Fetch https://news.ycombinator.com and tell me the top 5 stories on Hacker News right now

🔄 Initializing MCP client and LLM...
🚀 Running agent with query: Fetch https://news.ycombinator.com and tell me the top 5 stories on Hacker News right now
--------------------------------------------------
2026-02-28 19:31:32,878 - mcp_use - INFO - 🚀 Initializing MCP agent and connecting to services...


INFO:mcp_use:🚀 Initializing MCP agent and connecting to services...


2026-02-28 19:31:32,880 - mcp_use - INFO - 🔌 Found 0 existing sessions


INFO:mcp_use:🔌 Found 0 existing sessions


2026-02-28 19:31:32,883 - mcp_use - INFO - 🔄 No active sessions found, creating new ones...


...

2026-02-28 19:31:41,828 - mcp_use - INFO - ✅ Agent finished with output


INFO:mcp_use:✅ Agent finished with output


2026-02-28 19:31:41,832 - mcp_use - INFO - 🎉 Agent execution complete in 8.95 seconds


INFO:mcp_use:🎉 Agent execution complete in 8.95 seconds



🤖 Agent Response:

Here are the top 5 stories on Hacker News right now:

1. Cognitive Debt: When Velocity Exceeds Comprehension (365 points)
2. Obsidian Sync now has a headless client (172 points)
3. Verified Spec-Driven Development (VSDD) (59 points)
4. Addressing Antigravity Bans and Reinstating Access (139 points)
5. Woxi: Wolfram Mathematica Reimplementation in Rust (166 points)


In [ ]:
# Cell 9: Test the agent - Fetch a tech news page
query2 = "Fetch https://news.ycombinator.com and tell me the top 5 stories on Hacker News right now"
print(f"📝 Query: {query2}\n")

result2 = await run_agent_query(query2)

print(f"\n{'='*50}")
print(f"🤖 Agent Response:\n")
print(result2)

📝 Query: Fetch https://news.ycombinator.com and tell me the top 5 stories on Hacker News right now

🔄 Initializing MCP client and LLM...
🚀 Running agent with query: Fetch https://news.ycombinator.com and tell me the top 5 stories on Hacker News right now
--------------------------------------------------
2026-02-28 19:32:01,735 - mcp_use - INFO - 🚀 Initializing MCP agent and connecting to services...


INFO:mcp_use:🚀 Initializing MCP agent and connecting to services...


2026-02-28 19:32:01,738 - mcp_use - INFO - 🔌 Found 0 existing sessions


INFO:mcp_use:🔌 Found 0 existing sessions


2026-02-28 19:32:01,740 - mcp_use - INFO - 🔄 No active sessions found, creating new ones...


...

2026-02-28 19:32:10,176 - mcp_use - INFO - ✅ Agent finished with output


INFO:mcp_use:✅ Agent finished with output


2026-02-28 19:32:10,179 - mcp_use - INFO - 🎉 Agent execution complete in 8.44 seconds


INFO:mcp_use:🎉 Agent execution complete in 8.44 seconds



🤖 Agent Response:

Here are the top 5 stories on Hacker News right now:

1.  Cognitive Debt: When Velocity Exceeds Comprehension
2.  Obsidian Sync now has a headless client
3.  Verified Spec-Driven Development (VSDD)
4.  Addressing Antigravity Bans and Reinstating Access
5.  The happiest I've ever been


In [ ]:
# Cell 10: Test the agent - Airbnb search
query3 = "Search Airbnb for places in Tokyo for 2 adults"
print(f"📝 Query: {query3}\n")

result3 = await run_agent_query(query3)

print(f"\n{'='*50}")
print(f"🤖 Agent Response:\n")
print(result3)

📝 Query: Search Airbnb for places in Tokyo for 2 adults

🔄 Initializing MCP client and LLM...
🚀 Running agent with query: Search Airbnb for places in Tokyo for 2 adults
--------------------------------------------------
2026-02-28 19:51:51,677 - mcp_use - INFO - 🚀 Initializing MCP agent and connecting to services...


INFO:mcp_use:🚀 Initializing MCP agent and connecting to services...


2026-02-28 19:51:51,679 - mcp_use - INFO - 🔌 Found 0 existing sessions


INFO:mcp_use:🔌 Found 0 existing sessions


2026-02-28 19:51:51,681 - mcp_use - INFO - 🔄 No active sessions found, creating new ones...


...

2026-02-28 19:51:57,447 - mcp_use - INFO - ✅ Agent finished with output


INFO:mcp_use:✅ Agent finished with output


2026-02-28 19:51:57,451 - mcp_use - INFO - 🎉 Agent execution complete in 5.77 seconds


INFO:mcp_use:🎉 Agent execution complete in 5.77 seconds



🤖 Agent Response:

I am sorry, I encountered an error when trying to search for Airbnb listings in Tokyo. Please try again in some time.


In [ ]:
# Cell 11: Interactive mode - Ask your own questions
while True:
    user_input = input("\n🧑 You: ").strip()

    if user_input.lower() in {"exit", "quit", "stop"}:
        print("👋 Goodbye!")
        break

    if not user_input:
        print("⚠️ Please enter a question.")
        continue

    result = await run_agent_query(user_input)
    print(f"\n🤖 Agent: {result}")


🧑 You: What is the price of gold today
🔄 Initializing MCP client and LLM...
🚀 Running agent with query: What is the price of gold today
--------------------------------------------------
2026-02-28 19:52:31,554 - mcp_use - INFO - 🚀 Initializing MCP agent and connecting to services...


INFO:mcp_use:🚀 Initializing MCP agent and connecting to services...


2026-02-28 19:52:31,556 - mcp_use - INFO - 🔌 Found 0 existing sessions


INFO:mcp_use:🔌 Found 0 existing sessions


2026-02-28 19:52:31,561 - mcp_use - INFO - 🔄 No active sessions found, creating new ones...


...

2026-02-28 20:00:56,023 - mcp_use - INFO - ✅ Agent finished with output


INFO:mcp_use:✅ Agent finished with output


2026-02-28 20:00:56,026 - mcp_use - INFO - 🎉 Agent execution complete in 16.58 seconds


INFO:mcp_use:🎉 Agent execution complete in 16.58 seconds



🤖 Agent: The history of Artificial Intelligence (AI) can be understood through several key milestones:

1.  **The Birth of AI (1950s):** The field of AI was formally established at a workshop at Dartmouth College in 1956. Prior to this, pioneers like Alan Turing had already laid theoretical groundwork, proposing the Turing Test in 1950 as a measure of machine intelligence. Early AI research focused on simulating high-level human reasoning, leading to the development of programs that could play games, solve algebraic problems, and prove logical theorems.

2.  **The Era of Expert Systems (1970s-1980s):** Following initial optimism, AI research experienced a period known as the \"AI winter\" due to unmet expectations and funding cuts. However, the early 1980s saw a revival with the success of expert systems. These AI programs mimicked the knowledge and decision-making skills of human experts in specific domains, leading to commercial applications and renewed interest in the field.

3.  *